# Temperature, Pressure, and Chemistry

A TauREx model is assembled from interchangeable physical components. Before looking at spectra, it is worth pausing on the atmosphere itself: what temperature profile are we using, over what pressure range, and which gases are actually present?

This notebook is intentionally slower than the others. The goal is to build intuition for the objects that later get passed almost invisibly into transmission and emission models.

In [1]:
from _shared import build_base_components

context = build_base_components(download=False)
iso_t = context['iso_t']
press = context['press']
chemistry = context['chemistry']

print('Built a reusable atmospheric context.')

Built a reusable atmospheric context.


The helper constructs a complete atmospheric context in one step. That is convenient, but it also hides detail, so the rest of this notebook unpacks what came back and asks whether it makes physical sense.

The first inspection step is deliberately simple. Rather than jumping into plots immediately, we ask TauREx what kind of temperature and pressure objects we have constructed. This is often enough to catch configuration mistakes early.

In [2]:
iso_t, press

print(f'Layers: {press.nLayers}')
print(f'Pressure range: {press.profile.min():.3e} Pa to {press.profile.max():.3e} Pa')
print(f'Isothermal temperature: {iso_t.isoTemperature} K')

Layers: 100
Pressure range: 1.122e-05 Pa to 8.913e+04 Pa
Isothermal temperature: 2000.0 K


In [3]:
pressure_bar = press.profile / 1e5

print('First five pressure levels in bar:')
print(pressure_bar[:5])
print('Last five pressure levels in bar:')
print(pressure_bar[-5:])

First five pressure levels in bar:
[0.89125094 0.70794578 0.56234133 0.44668359 0.35481339]
Last five pressure levels in bar:
[2.81838293e-10 2.23872114e-10 1.77827941e-10 1.41253754e-10
 1.12201845e-10]


Chemistry is the next piece to inspect. TauREx separates gases that contribute explicit opacities from fill gases that mainly set the background atmosphere. That distinction matters later when interpreting which features show up in the spectrum.

In [4]:
print(f'Active gases: {chemistry.activeGases}')
print(f'Inactive gases: {chemistry.inactiveGases}')

for gas_name, mix_ratio in zip(chemistry.gases, chemistry.mixProfile[:, 0]):
    print(f'{gas_name}: {mix_ratio:.3e}')

Active gases: ('H2O', 'CH4', 'NH3', 'CO2')
Inactive gases: ('H2', 'He')
H2: 8.495e-01
He: 1.492e-01
H2O: 1.000e-03
CH4: 1.000e-04
NH3: 1.000e-04
CO2: 1.000e-04


In [5]:
layer0_total = chemistry.mixProfile[:, 0].sum()
print(f'Total mixing ratio in the first layer: {layer0_total:.6f}')

Total mixing ratio in the first layer: 1.000000


A useful cross-check is to ask whether the gas mixture is normalized the way you expect. Constant abundances are simple, but this habit becomes important once you move to more complex chemistry models.